# Polarizability by linear response

We compute the polarizability of a Helium atom. The polarizability
is defined as the change in dipole moment
$$
μ = ∫ r ρ(r) dr
$$
with respect to a small uniform electric field $E = -x$.

We compute this in two ways: first by finite differences (applying a
finite electric field), then by linear response. Note that DFTK is
not really adapted to isolated atoms because it uses periodic
boundary conditions. Nevertheless we can simply embed the Helium
atom in a large enough box (although this is computationally wasteful).

As in other tests, this is not fully converged, convergence
parameters were simply selected for fast execution on CI,

In [1]:
using DFTK
using LinearAlgebra
using PseudoPotentialData

a = 10.
lattice = a * I(3)  # cube of $a$ bohrs
pseudopotentials = PseudoFamily("cp2k.nc.sr.lda.v0_1.largecore.gth")
# Helium at the center of the box
atoms     = [ElementPsp(:He, pseudopotentials)]
positions = [[1/2, 1/2, 1/2]]


kgrid = [1, 1, 1]  # no k-point sampling for an isolated system
Ecut = 30
tol = 1e-8

# dipole moment of a given density (assuming the current geometry)
function dipole(basis, ρ)
    rr = [(r[1] - a/2) for r in r_vectors_cart(basis)]
    sum(rr .* ρ) * basis.dvol
end;

## Using finite differences
We first compute the polarizability by finite differences.
First compute the dipole moment at rest:

In [2]:
model  = model_DFT(lattice, atoms, positions;
                   functionals=LDA(), symmetries=false)
basis  = PlaneWaveBasis(model; Ecut, kgrid)
scfres = self_consistent_field(basis; tol)
μref   = dipole(basis, scfres.ρ)

n     Energy            log10(ΔE)   log10(Δρ)   Diag   Δtime
---   ---------------   ---------   ---------   ----   ------
  1   -2.770320866641                   -0.53    8.0    184ms
  2   -2.771683613570       -2.87       -1.31    1.0   95.1ms
  3   -2.771714146977       -4.52       -2.59    1.0    115ms
  4   -2.771714681198       -6.27       -3.77    1.0    101ms
  5   -2.771714714385       -7.48       -4.06    2.0    115ms
  6   -2.771714715224       -9.08       -5.48    1.0    107ms
  7   -2.771714715248      -10.61       -5.44    2.0    121ms
  8   -2.771714715250      -11.79       -6.36    1.0    111ms
  9   -2.771714715250      -13.05       -7.32    2.0    124ms
 10   -2.771714715250      -13.73       -7.43    1.0    114ms
 11   -2.771714715250   +  -13.77       -8.48    1.0    113ms


-0.00013457423567519926

Then in a small uniform field:

In [3]:
ε = .01
model_ε = model_DFT(lattice, atoms, positions;
                    functionals=LDA(),
                    extra_terms=[ExternalFromReal(r -> -ε * (r[1] - a/2))],
                    symmetries=false)
basis_ε = PlaneWaveBasis(model_ε; Ecut, kgrid)
res_ε   = self_consistent_field(basis_ε; tol)
με = dipole(basis_ε, res_ε.ρ)

n     Energy            log10(ΔE)   log10(Δρ)   Diag   Δtime
---   ---------------   ---------   ---------   ----   ------
  1   -2.770432759222                   -0.53    9.0    167ms
  2   -2.771771001740       -2.87       -1.31    1.0    119ms
  3   -2.771801492138       -4.52       -2.56    1.0    113ms
  4   -2.771802026678       -6.27       -3.57    1.0    103ms
  5   -2.771802073991       -7.33       -4.16    2.0    119ms
  6   -2.771802074457       -9.33       -5.42    1.0    108ms
  7   -2.771802074475      -10.74       -5.54    2.0    125ms
  8   -2.771802074476      -12.66       -5.65    1.0    110ms
  9   -2.771802074476      -12.34       -7.47    1.0    112ms
 10   -2.771802074476      -13.66       -7.30    3.0    211ms
 11   -2.771802074476   +  -14.45       -8.38    1.0    776ms


0.01761222034160663

In [4]:
polarizability = (με - μref) / ε

println("Reference dipole:  $μref")
println("Displaced dipole:  $με")
println("Polarizability :   $polarizability")

Reference dipole:  -0.00013457423567519926
Displaced dipole:  0.01761222034160663
Polarizability :   1.774679457728183


The result on more converged grids is very close to published results.
For example [DOI 10.1039/C8CP03569E](https://doi.org/10.1039/C8CP03569E)
quotes **1.65** with LSDA and **1.38** with CCSD(T).

## Using linear response

Now we use linear response (also known as density-functional perturbation theory)
to compute this analytically; we refer to standard
textbooks for the formalism. In the following, $χ_0$ is the
independent-particle polarizability, and $K$ the
Hartree-exchange-correlation kernel. We denote with $δV_{\rm ext}$ an external
perturbing potential (like in this case the uniform electric field).

In [5]:
# `δVext` is the potential from a uniform field interacting with the dielectric dipole
# of the density.
δVext = [-(r[1] - a/2) for r in r_vectors_cart(basis)]
δVext = cat(δVext; dims=4)

54×54×54×1 Array{Float64, 4}:
[:, :, 1, 1] =
  5.0       5.0       5.0       5.0      …   5.0       5.0       5.0
  4.81481   4.81481   4.81481   4.81481      4.81481   4.81481   4.81481
  4.62963   4.62963   4.62963   4.62963      4.62963   4.62963   4.62963
  4.44444   4.44444   4.44444   4.44444      4.44444   4.44444   4.44444
  4.25926   4.25926   4.25926   4.25926      4.25926   4.25926   4.25926
  4.07407   4.07407   4.07407   4.07407  …   4.07407   4.07407   4.07407
  3.88889   3.88889   3.88889   3.88889      3.88889   3.88889   3.88889
  3.7037    3.7037    3.7037    3.7037       3.7037    3.7037    3.7037
  3.51852   3.51852   3.51852   3.51852      3.51852   3.51852   3.51852
  3.33333   3.33333   3.33333   3.33333      3.33333   3.33333   3.33333
  ⋮                                      ⋱                      
 -3.33333  -3.33333  -3.33333  -3.33333  …  -3.33333  -3.33333  -3.33333
 -3.51852  -3.51852  -3.51852  -3.51852     -3.51852  -3.51852  -3.51852
 -3.7037   -3.7037 

Then:
$$
δρ = χ_0 δV = χ_0 (δV_{\rm ext} + K δρ),
$$
which implies
$$
δρ = (1-χ_0 K)^{-1} χ_0 δV_{\rm ext}.
$$
From this we identify the polarizability operator to be $χ = (1-χ_0 K)^{-1} χ_0$.
Numerically, we apply $χ$ to $δV = -x$ by solving a linear equation
(the Dyson equation) iteratively.

First we do this using the `DFTK.solve_ΩplusK_split`
function provided by DFTK,
which uses an adaptive Krylov subspace algorithm [^HS2025]:

[^HS2025]:
    M. F. Herbst and B. Sun.
    *Efficient Krylov methods for linear response in plane-wave electronic structure calculations.*
    [arXiv 2505.02319](http://arxiv.org/abs/2505.02319)

In [6]:
# Multiply δVext times the Bloch waves, then solve the Dyson equation:
δVψ = DFTK.multiply_ψ_by_blochwave(scfres.basis, scfres.ψ, δVext)
res = DFTK.solve_ΩplusK_split(scfres, δVψ; verbose=true)

Iter  Restart  Krydim  log10(res)  avg(CG)  Δtime   Comment
----  -------  ------  ----------  -------  ------  ---------------
                                      13.0  70.0ms  Non-interacting
   1        0       1       -0.60     10.0   1.10s  
   2        0       2       -2.42      7.0   178ms  
   3        0       3       -3.55      5.0   100ms  
   4        0       4       -5.33      4.0   120ms  
   5        0       5       -7.59      1.0  81.8ms  
   6        0       6      -10.12      1.0  85.3ms  
   7        1       1       -7.42     13.0   220ms  Restart
   8        1       2       -8.92      1.0  84.3ms  
                                      13.0   129ms  Final orbitals


(δψ = Matrix{ComplexF64}[[-0.0032357868327878513 - 0.0006265774212990251im 0.0 + 0.0im 0.0 + 0.0im 0.0 + 0.0im; -0.06136741154439899 + 0.30324946181558515im 0.0 + 0.0im 0.0 + 0.0im 0.0 + 0.0im; … ; -0.007969295201034166 + 0.04028445325166731im 0.0 + 0.0im 0.0 + 0.0im 0.0 + 0.0im; 0.015082549758482337 - 0.07952049147711347im 0.0 + 0.0im 0.0 + 0.0im 0.0 + 0.0im]], δρ = [-3.620694500603943e-7 -2.5020151100142644e-7 … -4.8806448004870225e-8 -2.502015277888342e-7; -6.293618278183234e-7 -4.793037617799348e-7 … -1.2077851756601706e-7 -4.793037777429526e-7; … ; 1.3626269781850336e-7 1.1356238821607444e-7 … 4.711997326574195e-8 1.1356237722500846e-7; 1.330548900614367e-7 1.363265138701202e-7 … 5.200689577125935e-8 1.3632649879050242e-7;;; -2.5020154889344787e-7 -1.7404910149715857e-7 … -3.556496418714071e-8 -1.7404911556542493e-7; -4.793037862285526e-7 -3.6797331333668695e-7 … -1.1202903171349465e-7 -3.679733315468922e-7; … ; 1.1356237189607182e-7 1.1211106579671134e-7 … 1.0980691596831576e-7 1

From the result of `solve_ΩplusK_split` we can easily compute the polarisabilities:

In [7]:
println("Non-interacting polarizability: $(dipole(basis, res.δρ0))")
println("Interacting polarizability:     $(dipole(basis, res.δρ))")

Non-interacting polarizability: 1.9257125255359082
Interacting polarizability:     1.7736548565307193


As expected, the interacting polarizability matches the finite difference
result. The non-interacting polarizability is higher.

### Manual solution of the Dyson equations
To see what goes on under the hood, we also show how to manually solve the
Dyson equation using KrylovKit:

In [8]:
using KrylovKit

# Apply $(1- χ_0 K)$
function dielectric_operator(δρ)
    δV = apply_kernel(basis, δρ; scfres.ρ)
    χ0δV = apply_χ0(scfres, δV).δρ
    δρ - χ0δV
end

# Apply $χ_0$ once to get non-interacting dipole
δρ_nointeract = apply_χ0(scfres, δVext).δρ

# Solve Dyson equation to get interacting dipole
δρ = linsolve(dielectric_operator, δρ_nointeract, verbosity=3)[1]

println("Non-interacting polarizability: $(dipole(basis, δρ_nointeract))")
println("Interacting polarizability:     $(dipole(basis, δρ))")

[ Info: GMRES linsolve starts with norm of residual = 4.19e+00
[ Info: GMRES linsolve in iteration 1; step 1: normres = 2.49e-01
[ Info: GMRES linsolve in iteration 1; step 2: normres = 3.77e-03
[ Info: GMRES linsolve in iteration 1; step 3: normres = 2.85e-04
[ Info: GMRES linsolve in iteration 1; step 4: normres = 4.69e-06
[ Info: GMRES linsolve in iteration 1; step 5: normres = 1.09e-08
[ Info: GMRES linsolve in iteration 1; step 6: normres = 6.28e-11
[ Info: GMRES linsolve in iteration 1; step 7: normres = 4.82e-10
[ Info: GMRES linsolve in iteration 2; step 1: normres = 5.38e-11
[ Info: GMRES linsolve in iteration 2; step 2: normres = 8.34e-12
┌ Info: GMRES linsolve converged at iteration 2, step 3:
│ * norm of residual = 2.12e-13
└ * number of operations = 12
Non-interacting polarizability: 1.9257125255358656
Interacting polarizability:     1.7736548576135913


We obtain the identical result to above.